# 6.3. Create distance measure

This notebook creates a geographical distance measure between lab groups, for use as a network proximity measure in the spillovers analysis.

Input:
- Cleaned rooms dataset (6_1 output): rooms are formatted "BUI-F-X" (building-floor-room)
- Building coordinates workbook

Output:
- Pairwise labgroupid distance dataset (km): 0 if the two lab groups have a room in the same
  building, otherwise the minimum distance between any of their buildings

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from itertools import combinations
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data
rooms = pd.read_csv(config.CLEAN_DATA / "rooms_cleaned.csv")

building_coords = pd.read_excel(
    config.CLEANING_WORKBOOKS / "building_list.xlsx"
)[["Building", "Latitude", "Longitude"]]

## (1) Extract each lab group's set of buildings

In [3]:
# Building is the text before the first "-" in "BUI-F-X" (a handful of rooms have no dash at
# all - str.split still returns the whole string as one "building")
rooms["building"] = rooms["room"].str.strip().str.split("-").str[0].str.strip()
buildings_by_lab = rooms.groupby("labgroupid")["building"].apply(set)

print(f"{len(buildings_by_lab)} lab groups have at least one room")
print(f"{rooms['building'].nunique()} distinct buildings reported")

135 lab groups have at least one room
35 distinct buildings reported


## (2) Compute building-to-building distances

In [4]:
# Haversine distance in km
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians(lat1), np.radians(lon1), np.radians(lat2), np.radians(lon2)
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

# A few buildings have no coordinates on file - report them; they still get distance 0 for a
# same-building match, but can't contribute to a cross-building minimum
missing_coords = building_coords[building_coords[["Latitude", "Longitude"]].isna().any(axis=1)]
print(f"{len(missing_coords)} buildings have no coordinates")

coords = building_coords.dropna(subset=["Latitude", "Longitude"]).set_index("Building")
bldg_dist = pd.DataFrame(
    haversine_km(
        coords["Latitude"].values[:, None], coords["Longitude"].values[:, None],
        coords["Latitude"].values[None, :], coords["Longitude"].values[None, :],
    ),
    index=coords.index, columns=coords.index,
)

3 buildings have no coordinates


## (3) Compute the labgroupid-pair distance

In [5]:
# 0 if the pair shares a building, else the minimum distance between any of their buildings
# (ignoring buildings with no coordinates on file). A shared building always gives distance 0
# even if it has no coordinates. Coordinates only matter when a pair shares no building at all;
# a lab whose only buildings are all coordinate-less would then get NaN against other
# labs, though this doesn't currently occur (0 pairs with no distance available below).
labgroupids = sorted(buildings_by_lab.index)
labgroupids = sorted(buildings_by_lab.index)
rows = []
for a, b in combinations(labgroupids, 2):
    buildings_a, buildings_b = buildings_by_lab[a], buildings_by_lab[b]
    if buildings_a & buildings_b:
        distance = 0.0
    else:
        valid_a = [x for x in buildings_a if x in bldg_dist.index]
        valid_b = [x for x in buildings_b if x in bldg_dist.index]
        pair_distances = [bldg_dist.loc[x, y] for x in valid_a for y in valid_b]
        distance = min(pair_distances) if pair_distances else np.nan
    rows.append((a, b, distance))

distance_df = pd.DataFrame(rows, columns=["labgroupid_a", "labgroupid_b", "distance_km"])

print(f"{len(distance_df)} lab group pairs")
print(f"{(distance_df['distance_km'] == 0).sum()} pairs share a building (distance = 0)")
print(f"{distance_df['distance_km'].isna().sum()} pairs have no computable distance")
print(distance_df["distance_km"].describe())

9045 lab group pairs
575 pairs share a building (distance = 0)
0 pairs have no computable distance
count    9045.000000
mean        1.252843
std         1.737564
min         0.000000
25%         0.118463
50%         0.227943
75%         2.254434
max         8.889042
Name: distance_km, dtype: float64


## (4) Save distance dataset

In [6]:
# Save cleaned dataset
distance_df.to_csv(config.CLEAN_DATA / "distances_cleaned.csv", index=False)